# 기본 53열 데이터와 확장 126열 데이터 비교

동일 학생 단위 3-Fold, 동일 Target, 동일 하이퍼파라미터에서 기본 Feature Set(모델 입력 51개)과 확장 Feature Set(모델 입력 124개)을 비교한다.

- 기본 데이터: `../data/oulad_weekly_next_week_base.csv`
- 확장 데이터: `../data/oulad_weekly_next_week.csv`
- 비교 모델: XGBoost, CatBoost
- 평가지표: PR-AUC, Recall@Top-20%, Precision@Top-20%, Brier Score, ECE
- 결과 저장: `../results/feature_set_comparison/`

> 실행 시간이 길면 `04_compare_base_vs_enhanced_features.py`의 `MODELS_TO_RUN`을 `['xgboost']`로 바꿔 1차 확인한 뒤 CatBoost를 추가 실행한다.

In [1]:
import runpy
from pathlib import Path

# 프로젝트 루트 또는 notebooks 폴더에서 실행해도 비교 스크립트를 찾도록 처리한다.
script_path = Path.cwd() / '04_compare_base_vs_enhanced_features.py'
if not script_path.exists():
    script_path = Path.cwd() / 'models' / 'notebooks' / '04_compare_base_vs_enhanced_features.py'
runpy.run_path(str(script_path), run_name='__main__')

공통 관측치 확인 완료: 895,005행

===== base_51_features =====
[xgboost] 입력 Feature 51개 학습 시작
  XGBoost Fold 1: PR-AUC=0.0863
  XGBoost Fold 2: PR-AUC=0.0816
  XGBoost Fold 3: PR-AUC=0.0802
[catboost] 입력 Feature 51개 학습 시작
  CatBoost Fold 1: PR-AUC=0.0919
  CatBoost Fold 2: PR-AUC=0.0919
  CatBoost Fold 3: PR-AUC=0.0867

===== enhanced_124_features =====
[xgboost] 입력 Feature 124개 학습 시작
  XGBoost Fold 1: PR-AUC=0.0904
  XGBoost Fold 2: PR-AUC=0.0869
  XGBoost Fold 3: PR-AUC=0.0805
[catboost] 입력 Feature 124개 학습 시작
  CatBoost Fold 1: PR-AUC=0.1012
  CatBoost Fold 2: PR-AUC=0.0961
  CatBoost Fold 3: PR-AUC=0.0910

=== 전체 OOF 비교 결과 ===
          feature_set    model   rows  feature_count  target_rate  pr_auc_average_precision  brier_score  ece_10bin  recall_at_top_20pct  precision_at_top_20pct  top_20pct_count  top_20pct_threshold
     base_51_features catboost 895005             51     0.007455                  0.089320     0.007066   0.000198             0.696193                0.025950           17

{'__name__': '__main__',
 '__doc__': '기본 53열 데이터와 확장 126열 데이터의 공정한 모델 성능 비교 코드.',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'C:\\SKN_AI\\skn_2nd_project\\oulad-churn-prediction\\models\\notebooks\\04_compare_base_vs_enhanced_features.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <functio

## 선택 기준

1. CatBoost 결과를 우선으로 보고, 확장 데이터가 PR-AUC와 Recall@Top-20%를 일관되게 개선하는지 확인한다.
2. Precision@Top-20%와 Brier·ECE가 크게 나빠지면 개입 인원과 확률 신뢰도를 함께 고려한다.
3. Fold별 지표가 한 Fold에서만 좋아진 결과인지 확인한다.
4. 확장 데이터를 채택한 뒤에만 RandomForest를 추가 비교하고, 최종 CatBoost에는 별도 Validation Set 기반 확률 보정을 적용한다.

In [4]:
import pandas as pd
pd.read_csv('../results/feature_set_comparison/base_vs_enhanced_metrics.csv')

,feature_set,model,rows,feature_count,target_rate,pr_auc_average_precision,brier_score,ece_10bin,recall_at_top_20pct,precision_at_top_20pct,top_20pct_count,top_20pct_threshold
0,base_51_features,catboost,895005,51,0.007455,0.089320,0.007066,0.000198,0.696193,0.025950,179001,0.010025
1,enhanced_124_features,catboost,895005,124,0.007455,0.094775,0.007045,0.000242,0.711331,0.026514,179001,0.009791
2,base_51_features,xgboost,895005,51,0.007455,0.080788,0.148658,0.299222,0.699341,0.026067,179001,0.544353
3,enhanced_124_features,xgboost,895005,124,0.007455,0.085072,0.139304,0.273862,0.724820,0.027017,179001,0.530692


## 결과
확장 데이터(124 feature)는 두 모델 모두에서:
- PR-AUC 상승
- 상위 위험군 20% 안에서 이탈자 포착률 상승
- 위험군 Precision 상승
- XGBoost의 Brier·ECE도 개선
#### 따라서 확장 데이터(124 feature) 선택